# 09a — Plugin smoke test: no model, one GPU, ~5 GB

Checks the `dg_topk_plugin` code paths **inside your installed vLLM** without starting a server or loading the model, so it can run on any GPU with ~5 GB free (GPUs 0/1 can stay busy).

1. vLLM's plugin loader finds and runs our plugin (`vllm.plugins.load_general_plugins()`), and it patches vLLM's DiffusionGemma module.
2. On real logits and the real embedding table, the patched `sample_row_stats` gives **identical** argmax / samples / entropies to vLLM's original (mode `vllm`), and the patched self-conditioning path (legacy store helper or current compiled-step adapter) gives a hint within ~0.1% of vLLM's full hint, with finished and pinned positions zeroed.
3. Mode `topk_exact`: identical argmax / entropies, valid sampled tokens.
4. A constrained step (logits over a small allowed set, as vLLM uses for structured reads) runs through the patched functions.

Run it in the `.venv-vllm` environment with the plugin installed (RUNBOOK 1b).

In [4]:
import os, sys, subprocess, importlib
if 'torch' in sys.modules:
    raise RuntimeError('Restart the kernel first.')
GPU = None      # None = the GPU with the most free memory
if GPU is None:
    rows = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.free', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True, check=True).stdout.strip().splitlines()
    GPU = str(max(((int(i), int(f)) for i, f in (r.split(', ') for r in rows)), key=lambda t: t[1])[0])
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ.pop('DG_TOPK_SC_K', None)          # off while we keep vLLM's original functions
print('using GPU', GPU)

import importlib.metadata as md_
import torch, vllm
import vllm.model_executor.models.diffusion_gemma as dg
sys.path.insert(0, os.path.abspath('./dg_topk_plugin'))
print('vllm', vllm.__version__)
eps = [e for e in md_.entry_points(group='vllm.general_plugins') if e.name == 'dg_topk']
print('plugin entry point registered with vLLM:', bool(eps))
assert eps, 'plugin not installed: pip install -e ./dg_topk_plugin'
orig_stats = dg.sample_row_stats
orig_store = getattr(dg, '_store_self_conditioning', None)
orig_step = getattr(dg, '_compiled_sample_step', None)

os.environ['DG_TOPK_SC_K'] = '64'
os.environ['DG_TOPK_SC_TAIL'] = 'renorm'
os.environ['DG_TOPK_SAMPLE'] = 'vllm'
from vllm.plugins import load_general_plugins
import vllm.plugins as vp
already = vp.plugins_loaded
load_general_plugins()                          # what every vLLM process does at start-up
if not getattr(dg, '_dg_topk_patched', False) and already:
    import dg_topk_plugin; dg_topk_plugin.register()      # plugins were already loaded in this process (with k unset)
print('plugin patched vLLM:', getattr(dg, '_dg_topk_patched', False), '| via vLLM loader:', not already)
assert getattr(dg, '_dg_topk_patched', False), 'the plugin did not patch vLLM'
assert dg.sample_row_stats is not orig_stats
if orig_store is not None:
    assert dg._store_self_conditioning is not orig_store
else:
    assert dg._compiled_sample_step is not orig_step
print('register_warmup kept on the patched function:', hasattr(dg.sample_row_stats, 'register_warmup'))

using GPU 6


/data/users/shreenis/private/VLA_experiments/.venv-vllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[dg_topk] top-k self-conditioning enabled: k=64, tail=renorm, sample=vllm (pid 3325458)


vllm 0.31.0
plugin entry point registered with vLLM: True
plugin patched vLLM: True | via vLLM loader: True
register_warmup kept on the patched function: False


In [5]:
import hint_bench as hb
CL = 256
saved = hb.load_saved_logits('./runs/nb04/step_*.pt')
assert saved, 'needs the logits saved by notebook 04 in runs/nb04/'
E, _ = hb.load_embedding_table('google/diffusiongemma-26B-A4B-it', 'cuda')
V, D = E.shape
n = 4
L, temps = hb.make_batch(saved, n, CL, 'cuda')
normalizer = torch.tensor(D ** 0.5, dtype=E.dtype, device='cuda')
seed = 1234

a1, s1, e1, p1 = orig_stats(L, temps, CL, seed, E.dtype)
a2, s2, e2, p2 = dg.sample_row_stats(L, temps, CL, seed, E.dtype)
print('mode vllm | argmax identical:', bool(torch.equal(a1, a2)), '| sample identical:', bool(torch.equal(s1, s2)),
      '| entropy max |diff|:', float((e1 - e2).abs().max()), '| probs type:', type(p2).__name__)

def run_store(store, probs, keep, pin):
    sc = torch.zeros(n, CL, D, dtype=E.dtype, device='cuda')
    slots = torch.arange(n, device='cuda')
    probs = probs.view(n, CL, -1)
    if store is None:
        soft = torch.matmul(probs.to(E.dtype), E)
        sc[slots] = (soft * normalizer * (keep[:, None, None] & ~pin[:, :, None])).to(sc.dtype)
    else:
        store(sc, slots, keep, probs, pin, E, normalizer, 0, V, 1, '')
    return sc.float()
keep = torch.tensor([True, True, False, True], device='cuda')           # slot 2 finishes: its hint must be zero
pin = torch.zeros(n, CL, dtype=torch.bool, device='cuda'); pin[0, :8] = True   # pinned positions: hint must be zero
if orig_step is None:
    h1 = run_store(orig_store, p1, keep, pin)
    h2 = run_store(dg._store_self_conditioning, p2, keep, pin)
else:
    def run_step(step, probs):
        slots = torch.arange(n, device='cuda')
        sc = torch.zeros(n, CL, D, dtype=E.dtype, device='cuda')
        phase = ~keep
        step(
            new_tokens=s1.view(n, CL), argmax_tokens=a1.view(n, CL),
            token_entropy=e1.view(n, CL), probs=probs.view(n, CL, -1),
            decode_slots=slots, decode_idx=slots, all_slots=slots,
            valid_canvas_len=torch.full((n,), CL, dtype=torch.long, device='cuda'),
            canvas=torch.zeros(n, CL, dtype=torch.long, device='cuda'),
            argmax_canvas=torch.zeros(n, CL, dtype=torch.long, device='cuda'),
            step_tensor=torch.zeros(n, dtype=torch.int32, device='cuda'),
            is_encoder_phase=phase, confident_tensor=torch.zeros(n, dtype=torch.bool, device='cuda'),
            sc_embeds=sc, embed_weight=E, normalizer=normalizer,
            history=torch.zeros(n, 3, CL, dtype=torch.long, device='cuda'),
            history_len_tensor=torch.zeros(n, dtype=torch.long, device='cuda'),
            max_steps_tensor=torch.full((n,), 100, dtype=torch.int32, device='cuda'),
            pin_mask=pin, seed_canvas=torch.zeros(n, CL, dtype=torch.long, device='cuda'),
            read_only=torch.zeros(n, dtype=torch.bool, device='cuda'),
            sampled=torch.zeros(n, CL, dtype=torch.long, device='cuda'),
            num_sampled=torch.zeros(n, dtype=torch.long, device='cuda'),
            draft_tokens=torch.zeros(n, CL, dtype=torch.long, device='cuda'),
            confidence_threshold=0.0, vocab_size=V, CL=CL, ST=3, entropy_bound=1.0,
            sc_vocab_start=0, sc_vocab_end=V, tp_size=1, tp_group_name='', compute_sc=True)
        return sc.float()
    h1 = run_step(orig_step, p1)
    h2 = run_step(dg._compiled_sample_step, p2)
live = keep[:, None] & ~pin
rel = (h2 - h1).norm(dim=-1)[live] / h1.norm(dim=-1)[live].clamp_min(1e-6)
print(f'hint vs vLLM full hint: mean {100 * float(rel.mean()):.3f}%  p99 {100 * float(rel.quantile(0.99)):.3f}%')
print('zero where vLLM writes zero (finished slot, pinned positions):', bool((h2[~live] == 0).all()), bool((h1[~live] == 0).all()))

mode vllm | argmax identical: True | sample identical: True | entropy max |diff|: 4.470348358154297e-07 | probs type: TopKProbs
hint vs vLLM full hint: mean 0.113%  p99 0.839%
zero where vLLM writes zero (finished slot, pinned positions): True True


In [6]:
# Re-register with a different sampling mode; the plugin retains vLLM's original functions.
os.environ['DG_TOPK_SAMPLE'] = 'topk_exact'
import dg_topk_plugin; dg_topk_plugin.register()
a3, s3, e3, p3 = dg.sample_row_stats(L, temps, CL, seed, E.dtype)
print('mode topk_exact | argmax identical:', bool(torch.equal(a1, a3)), '| entropy max |diff|:', float((e1 - e3).abs().max()),
      '| samples in vocab:', bool(((s3 >= 0) & (s3 < V)).all()),
      '| samples equal to vLLM\'s Gumbel samples (expected: mostly not):', f'{float((s3 == s1).float().mean()):.2f}')
a4, s4, e4, p4 = dg.sample_row_stats(L, temps, CL, seed, None)             # a step with no self-conditioning
print('no-self-conditioning step returns probs=None:', p4 is None)

# constrained step: logits over a small allowed set of 16 ids
Lc = L[:, :16].contiguous()
ac, sc_, ec, pc = dg.sample_row_stats(Lc, temps, CL, seed, E.dtype)
hc = run_store(dg._store_self_conditioning, pc, torch.ones(n, dtype=torch.bool, device='cuda'), torch.zeros(n, CL, dtype=torch.bool, device='cuda'))
print('constrained (16 ids): runs, samples in range:', bool(((sc_ >= 0) & (sc_ < 16)).all()), '| hint finite:', bool(torch.isfinite(hc).all()))
print('\nSmoke test finished. If every line above is True / small, the plugin is ready for notebook 09.')

[dg_topk] top-k self-conditioning enabled: k=64, tail=renorm, sample=topk_exact (pid 3325458)


mode topk_exact | argmax identical: True | entropy max |diff|: 4.470348358154297e-07 | samples in vocab: True | samples equal to vLLM's Gumbel samples (expected: mostly not): 0.86
no-self-conditioning step returns probs=None: True
constrained (16 ids): runs, samples in range: True | hint finite: True

Smoke test finished. If every line above is True / small, the plugin is ready for notebook 09.
